# Multi-Organ Drug Toxicity Screening

**Clinical Application:** Predictive toxicology and drug safety assessment

**Learning Objectives:**
1. Understand organ-on-chip modeling principles
2. Simulate multi-organ drug pharmacokinetics
3. Assess cardiotoxicity, hepatotoxicity, and immunotoxicity
4. Predict adverse drug reactions before clinical trials

**Clinical Relevance:**
- Drug development screening (FDA Modernization Act 2.0)
- Personalized medicine and pharmacogenomics
- Drug-drug interaction prediction
- Therapeutic window optimization

**Background:**

Traditional drug screening relies on animal models and late-stage clinical detection of toxicity. Organ-on-chip technology enables:
- **Early detection** of multi-organ toxicity
- **Mechanistic understanding** of adverse effects
- **Reduced animal testing** (3Rs: Replace, Reduce, Refine)
- **Patient-specific** toxicity prediction

**Key Toxicity Targets:**
1. **Cardiotoxicity**: hERG channel blockade → QT prolongation → Torsades de Pointes
2. **Hepatotoxicity**: CYP450 inhibition → drug accumulation → liver injury
3. **Immunotoxicity**: Cytokine storm → systemic inflammation

**References:**
- Ingber (2022) Nature Reviews Drug Discovery - Organs-on-chips
- Fermini et al. (2016) J Pharmacol Toxicol Methods - CiPA initiative
- FDA Modernization Act 2.0 (2022) - Alternatives to animal testing

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from src.organchip.orchestrator import OrganChipSuite
from src.organchip.pkpd import PKPDParameters

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 12)
print("✓ Imports successful")

## Part 1: Doxorubicin Cardiotoxicity

Doxorubicin is a highly effective chemotherapy agent with dose-limiting cardiotoxicity.

**Mechanism:**
- Accumulates in cardiac mitochondria
- Generates reactive oxygen species (ROS)
- Causes myocyte apoptosis and contractile dysfunction
- Risk increases with cumulative dose >400 mg/m²

In [ ]:
# Create organ chip suite
suite = OrganChipSuite()

# Test doxorubicin at clinical dose
print("="*60)
print("DOXORUBICIN CARDIOTOXICITY SCREENING")
print("="*60)
print("\nSimulating 48-hour exposure at 5 mg/kg IV...\n")

results_dox = suite.run_drug_test(
    drug_name="Doxorubicin",
    dose_mg_kg=5.0,          # Standard chemotherapy dose
    duration_hours=48.0,     # Typical monitoring period
    dt_minutes=1.0,          # 1-minute resolution
)

# Extract time series
times = np.array(results_dox['times_hours'])
drug_conc = np.array(results_dox['drug_concentration'])
cardiac_biomarkers = results_dox['cardiac_outputs']
hepatic_biomarkers = results_dox['hepatic_outputs']
immune_biomarkers = results_dox['immune_outputs']

# Create comprehensive toxicity dashboard
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(4, 2, hspace=0.3, wspace=0.3)

# Plot 1: Drug Concentration (PK)
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(times, drug_conc, 'purple', linewidth=2.5)
ax1.axhline(0.1, color='orange', linestyle='--', alpha=0.7, label='Therapeutic threshold')
ax1.axhline(1.0, color='red', linestyle='--', alpha=0.7, label='Toxic threshold')
ax1.set_xlabel('Time (hours)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Plasma Concentration (μM)', fontsize=11, fontweight='bold')
ax1.set_title('Pharmacokinetics: Doxorubicin 5 mg/kg IV', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# Plot 2: Cardiac Troponin (cardiotoxicity marker)
ax2 = fig.add_subplot(gs[1, 0])
troponin = [b['troponin'] for b in cardiac_biomarkers]
ax2.plot(times, troponin, 'r-', linewidth=2.5)
ax2.axhline(0.04, color='orange', linestyle='--', alpha=0.7, label='Upper normal limit')
ax2.fill_between(times, 0, 0.04, alpha=0.2, color='green', label='Normal range')
ax2.set_xlabel('Time (hours)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Troponin I (ng/mL)', fontsize=11, fontweight='bold')
ax2.set_title('Cardiac Injury Marker', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# Plot 3: Left Ventricular Ejection Fraction
ax3 = fig.add_subplot(gs[1, 1])
ef = [b['ejection_fraction'] * 100 for b in cardiac_biomarkers]  # Convert to percentage
ax3.plot(times, ef, 'b-', linewidth=2.5)
ax3.axhline(55, color='orange', linestyle='--', alpha=0.7, label='Lower normal limit')
ax3.fill_between(times, 55, 80, alpha=0.2, color='green', label='Normal range')
ax3.set_xlabel('Time (hours)', fontsize=11, fontweight='bold')
ax3.set_ylabel('LVEF (%)', fontsize=11, fontweight='bold')
ax3.set_title('Cardiac Function', fontsize=12, fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

# Plot 4: Hepatic Enzymes (ALT)
ax4 = fig.add_subplot(gs[2, 0])
alt = [b['alt'] for b in hepatic_biomarkers]
ax4.plot(times, alt, 'orange', linewidth=2.5)
ax4.axhline(40, color='red', linestyle='--', alpha=0.7, label='Upper normal limit')
ax4.fill_between(times, 0, 40, alpha=0.2, color='green', label='Normal range')
ax4.set_xlabel('Time (hours)', fontsize=11, fontweight='bold')
ax4.set_ylabel('ALT (U/L)', fontsize=11, fontweight='bold')
ax4.set_title('Hepatotoxicity Marker', fontsize=12, fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

# Plot 5: Bilirubin
ax5 = fig.add_subplot(gs[2, 1])
bilirubin = [b['bilirubin'] for b in hepatic_biomarkers]
ax5.plot(times, bilirubin, 'brown', linewidth=2.5)
ax5.axhline(1.2, color='red', linestyle='--', alpha=0.7, label='Upper normal limit')
ax5.fill_between(times, 0, 1.2, alpha=0.2, color='green', label='Normal range')
ax5.set_xlabel('Time (hours)', fontsize=11, fontweight='bold')
ax5.set_ylabel('Bilirubin (mg/dL)', fontsize=11, fontweight='bold')
ax5.set_title('Liver Function', fontsize=12, fontweight='bold')
ax5.legend(fontsize=9)
ax5.grid(True, alpha=0.3)

# Plot 6: Cytokines (IL-6)
ax6 = fig.add_subplot(gs[3, 0])
il6 = [b['il6'] for b in immune_biomarkers]
ax6.plot(times, il6, 'darkred', linewidth=2.5)
ax6.axhline(5.0, color='orange', linestyle='--', alpha=0.7, label='Inflammation threshold')
ax6.fill_between(times, 0, 5.0, alpha=0.2, color='green', label='Normal range')
ax6.set_xlabel('Time (hours)', fontsize=11, fontweight='bold')
ax6.set_ylabel('IL-6 (pg/mL)', fontsize=11, fontweight='bold')
ax6.set_title('Inflammatory Response', fontsize=12, fontweight='bold')
ax6.legend(fontsize=9)
ax6.grid(True, alpha=0.3)

# Plot 7: Toxicity Scores (Radar)
ax7 = fig.add_subplot(gs[3, 1], projection='polar')
toxicity_scores = results_dox['toxicity_scores']
categories = list(toxicity_scores.keys())
values = list(toxicity_scores.values())

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
values += values[:1]
angles += angles[:1]

ax7.plot(angles, values, 'o-', linewidth=2.5, color='red', label='Doxorubicin')
ax7.fill(angles, values, alpha=0.25, color='red')
ax7.set_xticks(angles[:-1])
ax7.set_xticklabels(categories, fontsize=10)
ax7.set_ylim(0, 1)
ax7.set_yticks([0.25, 0.5, 0.75, 1.0])
ax7.set_yticklabels(['0.25', '0.5', '0.75', '1.0'], fontsize=8)
ax7.set_title('Multi-Organ Toxicity Profile', fontsize=12, fontweight='bold', pad=20)
ax7.axhline(0.5, color='orange', linestyle='--', alpha=0.5, label='Concern threshold')
ax7.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=9)
ax7.grid(True)

plt.suptitle('Doxorubicin Multi-Organ Toxicity Assessment', fontsize=15, fontweight='bold', y=0.995)
plt.show()

# Print summary report
print("\n" + "="*60)
print("TOXICITY ASSESSMENT REPORT")
print("="*60)
print(f"\nDrug: Doxorubicin")
print(f"Dose: 5.0 mg/kg IV")
print(f"Duration: 48 hours")
print(f"\nToxicity Scores (0-1, higher = more toxic):")
for organ, score in toxicity_scores.items():
    status = "⚠️ CONCERN" if score > 0.5 else "✓ Acceptable"
    print(f"  {organ.capitalize():15s}: {score:.3f}  {status}")

print(f"\nPeak Troponin I: {max(troponin):.3f} ng/mL (normal <0.04)")
print(f"Minimum LVEF: {min(ef):.1f}% (normal >55%)")
print(f"Peak ALT: {max(alt):.1f} U/L (normal <40)")
print(f"Peak IL-6: {max(il6):.1f} pg/mL (normal <5)")

print(f"\n\nCLINICAL INTERPRETATION:")
print(f"  - Significant cardiotoxicity detected (troponin elevation)")
print(f"  - Dose-dependent reduction in LVEF")
print(f"  - Mild hepatotoxicity (transient ALT elevation)")
print(f"  - Recommendation: Cardiac monitoring during treatment")
print(f"  - Consider cardioprotective agents (dexrazoxane)")
print("="*60)

## Part 2: hERG Channel Screening - QT Prolongation Risk

The hERG (human Ether-à-go-go-Related Gene) potassium channel is responsible for cardiac repolarization. Drug-induced hERG blockade is the leading cause of QT prolongation and sudden cardiac death.

**Clinical Context:**
- Leading cause of drug withdrawal from market
- FDA requires hERG screening for all new drugs (ICH S7B guideline)
- Examples: terfenadine, cisapride, sertindole (all withdrawn)

In [ ]:
# Test multiple drugs for hERG liability
test_drugs = [
    {"name": "Verapamil", "dose": 0.12, "ic50_hERG": 0.15},      # Calcium channel blocker (moderate hERG block)
    {"name": "Azithromycin", "dose": 0.5, "ic50_hERG": 56.0},   # Antibiotic (weak hERG block)
    {"name": "Ondansetron", "dose": 0.032, "ic50_hERG": 0.5},   # Antiemetic (strong hERG block)
    {"name": "Safe_Compound", "dose": 1.0, "ic50_hERG": 100.0}, # Hypothetical safe drug
]

results_collection = []

print("\n" + "="*60)
print("hERG CHANNEL SCREENING PANEL")
print("="*60)
print("\nTesting 4 compounds for cardiac safety...\n")

for drug in test_drugs:
    # Create custom PK/PD parameters
    pkpd = PKPDParameters(
        volume_distribution=1.0,
        clearance_rate=0.15,
        protein_binding=0.9,
        ic50_hERG=drug['ic50_hERG'],
    )
    
    suite_custom = OrganChipSuite(pkpd_params=pkpd)
    
    result = suite_custom.run_drug_test(
        drug_name=drug['name'],
        dose_mg_kg=drug['dose'],
        duration_hours=24.0,
        dt_minutes=1.0,
    )
    
    results_collection.append({
        'name': drug['name'],
        'result': result,
        'ic50': drug['ic50_hERG']
    })

# Visualize comparative hERG liability
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

colors = ['red', 'orange', 'darkred', 'green']

for idx, drug_data in enumerate(results_collection):
    ax = axes[idx]
    result = drug_data['result']
    times = np.array(result['times_hours'])
    
    # Extract QT interval proxy (APD from cardiac model)
    cardiac_outs = result['cardiac_outputs']
    apd = [b.get('apd90', 320) for b in cardiac_outs]  # APD90 in ms
    
    ax.plot(times, apd, color=colors[idx], linewidth=2.5)
    ax.axhline(320, color='gray', linestyle='--', alpha=0.5, label='Baseline APD')
    ax.axhline(450, color='red', linestyle='--', alpha=0.7, label='Dangerous prolongation')
    ax.fill_between(times, 250, 450, alpha=0.1, color='yellow')
    
    ax.set_xlabel('Time (hours)', fontsize=11, fontweight='bold')
    ax.set_ylabel('APD90 (ms)', fontsize=11, fontweight='bold')
    ax.set_title(f"{drug_data['name']}\n(hERG IC50 = {drug_data['ic50']:.2f} μM)", 
                fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([250, 500])
    
    # Add safety classification
    max_apd = max(apd)
    if max_apd > 450:
        safety = "HIGH RISK"
        box_color = 'red'
    elif max_apd > 400:
        safety = "MODERATE RISK"
        box_color = 'orange'
    else:
        safety = "LOW RISK"
        box_color = 'green'
    
    ax.text(0.95, 0.95, safety, transform=ax.transAxes, 
           fontsize=11, fontweight='bold', va='top', ha='right',
           bbox=dict(boxstyle='round', facecolor=box_color, alpha=0.7, edgecolor='black'))

plt.suptitle('Comparative hERG Channel Screening - QT Prolongation Risk', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print ranking table
print("\n" + "="*60)
print("hERG SCREENING RESULTS - RANKED BY CARDIAC RISK")
print("="*60)
print(f"\n{'Drug':<20} {'IC50 (μM)':<12} {'Max APD (ms)':<15} {'Risk Level':<15}")
print("-" * 60)

# Sort by cardiac toxicity score
ranked = sorted(results_collection, 
               key=lambda x: x['result']['toxicity_scores']['cardiac'], 
               reverse=True)

for drug_data in ranked:
    result = drug_data['result']
    cardiac_score = result['toxicity_scores']['cardiac']
    apd = [b.get('apd90', 320) for b in result['cardiac_outputs']]
    max_apd = max(apd)
    
    if cardiac_score > 0.7:
        risk = "⚠️  HIGH"
    elif cardiac_score > 0.4:
        risk = "⚡ MODERATE"
    else:
        risk = "✓ LOW"
    
    print(f"{drug_data['name']:<20} {drug_data['ic50']:<12.2f} {max_apd:<15.1f} {risk:<15}")

print("\n" + "="*60)
print("REGULATORY GUIDANCE (FDA ICH S7B):")
print("  - IC50 < 1 μM: High risk, requires thorough QT study")
print("  - IC50 1-10 μM: Moderate risk, clinical ECG monitoring")
print("  - IC50 > 30 μM: Low risk, standard monitoring")
print("="*60)

## Part 3: Drug-Drug Interaction - CYP450 Inhibition

Many adverse drug events result from pharmacokinetic interactions via cytochrome P450 (CYP450) enzyme inhibition.

**Example:** Ketoconazole (strong CYP3A4 inhibitor) + Simvastatin → Elevated statin levels → Rhabdomyolysis

In [ ]:
# Simulate drug-drug interaction
print("\n" + "="*60)
print("DRUG-DRUG INTERACTION SCREENING")
print("="*60)
print("\nScenario: Simvastatin (substrate) + Ketoconazole (CYP3A4 inhibitor)\n")

# Baseline: Simvastatin alone
pkpd_simvastatin = PKPDParameters(
    volume_distribution=1.2,
    clearance_rate=0.25,      # Normal hepatic clearance
    protein_binding=0.95,
)

suite_baseline = OrganChipSuite(pkpd_params=pkpd_simvastatin)
result_simva_alone = suite_baseline.run_drug_test(
    drug_name="Simvastatin",
    dose_mg_kg=0.4,          # 40 mg dose
    duration_hours=72.0,
    dt_minutes=5.0,
)

# With CYP3A4 inhibition (ketoconazole co-administration)
pkpd_inhibited = PKPDParameters(
    volume_distribution=1.2,
    clearance_rate=0.05,      # 80% reduction in clearance (strong inhibition)
    protein_binding=0.95,
)

suite_inhibited = OrganChipSuite(pkpd_params=pkpd_inhibited)
result_simva_keto = suite_inhibited.run_drug_test(
    drug_name="Simvastatin + Ketoconazole",
    dose_mg_kg=0.4,
    duration_hours=72.0,
    dt_minutes=5.0,
)

# Compare pharmacokinetics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

times_alone = np.array(result_simva_alone['times_hours'])
conc_alone = np.array(result_simva_alone['drug_concentration'])
times_combo = np.array(result_simva_keto['times_hours'])
conc_combo = np.array(result_simva_keto['drug_concentration'])

# Plot 1: Plasma concentration comparison
axes[0, 0].plot(times_alone, conc_alone, 'b-', linewidth=2.5, label='Simvastatin alone')
axes[0, 0].plot(times_combo, conc_combo, 'r-', linewidth=2.5, label='+ Ketoconazole (CYP3A4 inhibitor)')
axes[0, 0].axhline(0.01, color='orange', linestyle='--', alpha=0.7, label='Therapeutic level')
axes[0, 0].axhline(0.1, color='red', linestyle='--', alpha=0.7, label='Toxicity threshold')
axes[0, 0].set_xlabel('Time (hours)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Plasma Concentration (μM)', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Pharmacokinetic Interaction', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_yscale('log')

# Plot 2: Hepatic ALT (hepatotoxicity)
alt_alone = [b['alt'] for b in result_simva_alone['hepatic_outputs']]
alt_combo = [b['alt'] for b in result_simva_keto['hepatic_outputs']]
axes[0, 1].plot(times_alone, alt_alone, 'b-', linewidth=2.5, label='Simvastatin alone')
axes[0, 1].plot(times_combo, alt_combo, 'r-', linewidth=2.5, label='+ Ketoconazole')
axes[0, 1].axhline(40, color='orange', linestyle='--', alpha=0.7, label='Upper normal')
axes[0, 1].axhline(120, color='red', linestyle='--', alpha=0.7, label='3x ULN (hepatotoxicity)')
axes[0, 1].set_xlabel('Time (hours)', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('ALT (U/L)', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Hepatotoxicity Risk', fontsize=12, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: AUC comparison (exposure)
auc_alone = np.trapz(conc_alone, times_alone)
auc_combo = np.trapz(conc_combo, times_combo)
auc_ratio = auc_combo / auc_alone

drugs = ['Simvastatin\nAlone', 'Simvastatin +\nKetoconazole']
aucs = [auc_alone, auc_combo]
colors_bar = ['blue', 'red']

bars = axes[1, 0].bar(drugs, aucs, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=2)
axes[1, 0].set_ylabel('AUC (μM·h)', fontsize=11, fontweight='bold')
axes[1, 0].set_title(f'Drug Exposure (AUC Ratio = {auc_ratio:.1f}x)', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.2f}',
                   ha='center', va='bottom', fontsize=11, fontweight='bold')

# Plot 4: Toxicity comparison (radar)
ax_radar = fig.add_subplot(2, 2, 4, projection='polar')

toxicity_alone = result_simva_alone['toxicity_scores']
toxicity_combo = result_simva_keto['toxicity_scores']

categories = list(toxicity_alone.keys())
values_alone = list(toxicity_alone.values())
values_combo = list(toxicity_combo.values())

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
values_alone += values_alone[:1]
values_combo += values_combo[:1]
angles += angles[:1]

ax_radar.plot(angles, values_alone, 'o-', linewidth=2.5, color='blue', label='Alone', alpha=0.7)
ax_radar.fill(angles, values_alone, alpha=0.15, color='blue')
ax_radar.plot(angles, values_combo, 'o-', linewidth=2.5, color='red', label='+ Ketoconazole', alpha=0.7)
ax_radar.fill(angles, values_combo, alpha=0.15, color='red')

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(categories, fontsize=10)
ax_radar.set_ylim(0, 1)
ax_radar.set_yticks([0.25, 0.5, 0.75, 1.0])
ax_radar.set_yticklabels(['0.25', '0.5', '0.75', '1.0'], fontsize=8)
ax_radar.set_title('Multi-Organ Toxicity Comparison', fontsize=12, fontweight='bold', pad=20)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=10)
ax_radar.grid(True)

plt.suptitle('Drug-Drug Interaction: CYP450 Inhibition Impact', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print clinical summary
print("\n" + "="*60)
print("DRUG-DRUG INTERACTION ASSESSMENT")
print("="*60)
print(f"\nPharmacokinetic Changes:")
print(f"  AUC Increase: {auc_ratio:.1f}x (due to CYP3A4 inhibition)")
print(f"  Peak Concentration: {max(conc_combo)/max(conc_alone):.1f}x higher")
print(f"  Half-life: ~{auc_ratio:.1f}x prolonged")

print(f"\nToxicity Impact:")
print(f"  Hepatotoxicity: {toxicity_combo['hepatic']:.3f} vs {toxicity_alone['hepatic']:.3f} (alone)")
print(f"  Peak ALT: {max(alt_combo):.1f} vs {max(alt_alone):.1f} U/L")

print(f"\nClinical Recommendations:")
if auc_ratio > 3:
    print(f"  ⚠️  CONTRAINDICATED: {auc_ratio:.1f}x increase in exposure")
    print(f"  - Do NOT co-administer")
    print(f"  - Consider alternative therapy (pravastatin - not CYP3A4 substrate)")
elif auc_ratio > 2:
    print(f"  ⚠️  DOSE ADJUSTMENT REQUIRED")
    print(f"  - Reduce simvastatin dose to 10 mg (75% reduction)")
    print(f"  - Monitor hepatic enzymes weekly")
    print(f"  - Watch for myalgia/rhabdomyolysis symptoms")
else:
    print(f"  ✓ Use with caution")
    print(f"  - Standard dosing acceptable with monitoring")

print(f"\nReference: FDA Drug Interaction Guidance (2020)")
print("="*60)

## Summary and Future Directions

### Key Takeaways

**1. Organ-on-Chip Advantages:**
- Early detection of multi-organ toxicity
- Mechanistic understanding (vs black-box animal models)
- Reduced cost and time in drug development
- Ethical alternative to animal testing

**2. Critical Toxicity Screens:**
- **Cardiotoxicity**: hERG channel, troponin, LVEF
- **Hepatotoxicity**: ALT/AST, bilirubin, CYP450 function
- **Drug-Drug Interactions**: CYP450 inhibition, AUC changes

**3. Clinical Applications:**
- Preclinical drug screening (IND submission)
- Personalized medicine (patient-specific iPSC models)
- Drug repurposing safety assessment
- Biomarker discovery

**4. Regulatory Acceptance:**
- FDA Modernization Act 2.0 (2022) - Allows alternatives to animal testing
- ICH S7B/E14 - hERG and QT guidelines
- CiPA (Comprehensive in vitro Proarrhythmia Assay) initiative

### Future Directions

**Technological Advances:**
- Multi-organ-on-chip with perfusion (body-on-a-chip)
- Integration with microfluidics and biosensors
- Patient-derived iPSC cells for personalized screening
- AI/ML for toxicity prediction

**Research Opportunities:**
- Chronic toxicity assessment (weeks to months)
- Immunotoxicity and cytokine release syndrome
- Neurotoxicity screening
- Developmental and reproductive toxicity (DART)

### References

- Ingber DE (2022) Human organs-on-chips for disease modelling, drug development and personalized medicine. *Nat Rev Drug Discov* 21:467-491
- Fermini B et al. (2016) A new perspective in the field of cardiac safety testing through the comprehensive in vitro proarrhythmia assay paradigm. *J Pharmacol Toxicol Methods* 81:15-20
- FDA Modernization Act 2.0 (2022) S.5002 - 117th Congress
- Sager PT et al. (2014) Rechanneling the cardiac proarrhythmia safety paradigm: A meeting report from the Cardiac Safety Research Consortium. *Am Heart J* 167:292-300

---
© 2025 Multi-Heart-Model Project | MIT License